<a href="https://colab.research.google.com/github/maazthecoder/cosmic-planet-predictor/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import io
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [7]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
train = pd.read_csv('/content/drive/MyDrive/planet-dataset.csv')
train.head()

,id,Surface Temperature,Water Content,Gravity,Proximity to Star,Surface Pressure,Probe ID,Orbital Period,Meteor Impact Rate,Atmospheric Density,Atmospheric Composition Index,Stellar Luminosity,Magnetic Field Strength,Radiation Levels,Mineral Abundance,Moon Count,Prediction
0,21130,1.282054,0.332456,-0.279615,0.590274,2.996759,PB-579_C,0.502441,-2.102058,5.105846,0.396444,196.321476,Category_8,Category_14,-2.619419,3,0.0
1,23642,-0.982315,-1.416155,NaN,0.393591,NaN,PB-117_G,0.768882,NaN,-1.924974,0.587680,116.877774,Category_8,Category_8,1.390242,3,8.0
2,462,-1.460140,-0.049144,-3.569419,0.292521,0.282381,PB-359_I,-1.643907,0.954337,0.576191,-1.564693,99.154238,Category_9,Category_11,0.200713,1,3.0
3,19232,0.532329,NaN,2.079739,1.420518,2.993805,PB-902_N,-0.873525,-1.242220,2.643606,1.299399,228.059479,Category_10,Category_16,-2.311188,6,0.0
4,10016,NaN,0.316110,1.886688,-1.335994,NaN,PB-226_Y,2.550948,1.112678,-2.992624,-0.399022,NaN,Category_9,Category_11,1.175706,1,1.0


Initial Data Exploration


In [9]:
train = train.drop(columns=["id", "Probe ID"])

In [10]:
'''
other_cols = []
numerical_cols = []

for col in X.columns:
    if X[col].dtype == "int64" or X[col].dtype == "float64":    # We use dtype instead of type() because pandas columns have their own data types
        numerical_cols.append(col)
    else:
        other_cols.append(col)
'''

'\nother_cols = []\nnumerical_cols = []\n\nfor col in X.columns:\n    if X[col].dtype == "int64" or X[col].dtype == "float64":    # We use dtype instead of type() because pandas columns have their own data types\n        numerical_cols.append(col)\n    else:\n        other_cols.append(col)\n'

In [11]:
other_cols = ["Magnetic Field Strength", "Radiation Levels"]

In [12]:
for col in other_cols:
    train[col] = (
        train[col]
        .astype(str)
        .str.extract(r"Category_(\d+)")
        .astype(float)
    )

In [13]:
train[other_cols].isnull().mean()
train[other_cols].describe()

,Magnetic Field Strength,Radiation Levels
count,43256.000000,43262.000000
mean,9.437881,8.770445
std,2.659261,2.236586
min,1.000000,1.000000
25%,8.000000,7.000000
50%,9.000000,9.000000
75%,11.000000,10.000000
max,20.000000,20.000000


In [14]:
train['Radiation Levels']

,Radiation Levels
0,14.0
1,8.0
2,11.0
3,16.0
4,11.0
...,...
45563,9.0
45564,10.0
45565,10.0
45566,NaN


In [15]:
train[other_cols].isna().sum()


,0
Magnetic Field Strength,2312
Radiation Levels,2306


In [16]:
X = train.drop("Prediction", axis=1)
y = train["Prediction"]

In [17]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=64,
    stratify=y
)

In [18]:
missing_frac = X_train.isnull().mean()
high_missing_cols = missing_frac[missing_frac > 0.95].index.tolist()

X_train = X_train.drop(columns=high_missing_cols)
X_val   = X_val.drop(columns=high_missing_cols)

In [19]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("variance", VarianceThreshold(threshold=1e-4))
])

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, X_train.columns)
    ]
)

In [26]:
from sklearn.ensemble import HistGradientBoostingClassifier
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(
        max_depth=10,
        learning_rate=0.05,
        max_iter=300,
        min_samples_leaf=20,
        l2_regularization=0.1,
        random_state=64
    ))
])


In [27]:
model.fit(X_train, y_train)
y_pred = model.predict(X_val)

In [28]:
("Accuracy:", accuracy_score(y_val, y_pred))
#print("\nClassification Report:\n", classification_report(y_val, y_pred))
#print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))

('Accuracy:', 0.8926925608953259)